In [65]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text 

In [66]:
df = pd.read_csv(r'D:\Mohamed Refaat\DEPI\Project implemention\Intelligent-Support-Ticket\Project Implementation\Data\clean_data.csv')
df.head()

,created_at,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,platform,region,date
0,2024-01-31T05:14:27,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0,very_negative,1,android,EU,2024-01-31 05:14:27
1,2024-10-20T06:15:49,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is r...,Ticket closed without further action after no ...,238.32,0,neutral,3,web,other,2024-10-20 06:15:49
2,2023-08-27T16:08:33,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,61.32,0,very_negative,2,web,other,2023-08-27 16:08:33
3,2024-09-24T14:04:53,education,email,mobile_app,performance,low,closed_no_action,gold,Queries in the mobile app module are timing out.,We understand performance issues can be frustr...,Ticket closed without further action after no ...,226.93,0,positive,5,ios,other,2024-09-24 14:04:53
4,2024-11-01T16:14:51,education,in_app,billing,security_concern,urgent,resolved,gold,I noticed a suspicious login on my account.,We take security very seriously. This has been...,"Investigated logs, mitigated the issue, and up...",4.36,0,negative,3,api_client,APAC,2024-11-01 16:14:51


In [67]:
X = df['initial_message']
y = df['customer_sentiment']

In [68]:
from sklearn.model_selection import train_test_split

In [69]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [70]:
from sklearn.preprocessing import LabelEncoder

In [71]:
label = LabelEncoder()

In [72]:
y_train=label.fit_transform(y_train)
y_test=label.transform(y_test)


In [73]:
y_train

array([1, 2, 3, ..., 3, 4, 3])

In [74]:
preprocess = hub.KerasLayer(
    "https://tfhub.dev/tensorflow/albert_en_preprocess/3",
    name="preprocessing"
)

In [75]:
encoder = hub.KerasLayer(
    "https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-2_H-128_A-2/2",
    trainable=True,
    name="bert"
)


In [76]:
df['customer_sentiment'].unique()

array(['very_negative', 'neutral', 'positive', 'negative',
       'very_positive'], dtype=object)

In [77]:
text_input = tf.keras.layers.Input(shape=(),dtype=tf.string,name='text')

x = preprocess(text_input)

x = encoder(x)

x = x['pooled_output']

x = tf.keras.layers.Dense(32,activation='relu')(x)

x = tf.keras.layers.Dropout(0.3)(x)

output = tf.keras.layers.Dense(5,activation = 'softmax') (x)

In [78]:
model = tf.keras.Model(inputs=text_input,outputs=output)

In [79]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [80]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',      # metric to watch
    patience=3,              # stop if no improvement after 3 epochs
    restore_best_weights=True
)

In [81]:
history = model.fit(
    X_train,        # your training X (text)
    y_train,       # your training y (integer labels)
    validation_data=(X_test, y_test),  # validation X & y
    epochs=10,          # max epochs
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/10
 214/1503 [===>..........................] - ETA: 12:45 - loss: 1.6839 - accuracy: 0.2263

KeyboardInterrupt: 

In [ ]:
model.evaluate(X_test,y_test)

376/376 [==============================] - 75s 197ms/step - loss: 1.4549 - accuracy: 0.3783


[1.4548656940460205, 0.37827497720718384]